# Regresión Logística
## Predicción de Mortalidad Hospitalaria - MIMIC-III

**Dataset:** MIMIC-III (Medical Information Mart for Intensive Care)  
**Fuente:** https://mimic.mit.edu/

## Variable objetivo (Y)
`hospital_expire_flag` — Si el paciente falleció en el hospital (1=Sí, 0=No)

## Features seleccionadas (n=6)

| # | Feature | Descripción | Justificación |
|---|---|---|---|
| 1 | `admission_type_encoded` | Tipo de admisión codificado | Emergencias tienen mayor riesgo |
| 2 | `age` | Edad del paciente en años | Pacientes mayores tienen mayor riesgo |
| 3 | `gender_encoded` | Género codificado (0=F, 1=M) | Factor demográfico relevante |
| 4 | `los_days` | Días de estancia hospitalaria | Estancias largas pueden indicar gravedad |
| 5 | `has_chartevents_data` | Si tiene datos de eventos | Indica monitoreo intensivo |
| 6 | `insurance_encoded` | Tipo de seguro codificado | Relacionado con acceso a atención |

## Columnas descartadas

| Columna | Razón |
|---|---|
| `row_id`, `subject_id`, `hadm_id` | Identificadores — sin relación con mortalidad |
| `admittime`, `dischtime`, `deathtime` | Fechas — se usa para calcular los_days |
| `edregtime`, `edouttime` | Tiempos de emergencia — muchos nulos |
| `diagnosis` | Texto libre — requiere NLP |
| `discharge_location` | Filtrado de información — contiene el outcome |

## Modelo a implementar
**Regresión Logística** para clasificación binaria

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot

## 1. Carga del Dataset

Se cargan las tablas ADMISSIONS y PATIENTS.
Se combinan por subject_id para obtener información del paciente.
Se calculan features derivadas como edad y días de estancia.

In [ ]:
# Cargar datasets
admissions = pd.read_csv('ADMISSIONS.csv')
patients = pd.read_csv('PATIENTS.csv')

# Combinar tablas
data = admissions.merge(patients, on='subject_id', how='left')

# Convertir fechas
data['admittime'] = pd.to_datetime(data['admittime'])
data['dischtime'] = pd.to_datetime(data['dischtime'])
data['dob'] = pd.to_datetime(data['dob'])

# Calcular edad al momento de admisión
data['age'] = (data['admittime'] - data['dob']).dt.days / 365.25
data['age'] = data['age'].clip(lower=0, upper=90)  # Limitar edad máxima

# Calcular días de estancia
data['los_days'] = (data['dischtime'] - data['admittime']).dt.days

# Codificar variables categóricas
data['gender_encoded'] = (data['gender'] == 'M').astype(int)
data['admission_type_encoded'] = data['admission_type'].map({
    'EMERGENCY': 3, 'URGENT': 2, 'ELECTIVE': 1, 'NEWBORN': 0
}).fillna(0).astype(int)
data['insurance_encoded'] = data['insurance'].astype('category').cat.codes

# Definir features y variable objetivo
FEATURES = [
    'admission_type_encoded',
    'age',
    'gender_encoded',
    'los_days',
    'has_chartevents_data',
    'insurance_encoded'
]

TARGET = 'hospital_expire_flag'

# Extraer X e y
X = data[FEATURES].values.astype(np.float64)
y = data[TARGET].values.astype(np.float64)
m = y.size

print(f'Dimensiones dataset : {data.shape}')
print(f'X shape             : {X.shape}')
print(f'y shape             : {y.shape}')
print(f'm                   : {m:,}')
print(f'n features          : {len(FEATURES)}')
print(f'\nDistribución target:')
print(data[TARGET].value_counts())

## 2. Exploración del Dataset

Se revisan estadísticas básicas para entender la distribución
de los datos antes de entrenar el modelo.

In [ ]:
print('=== Información general ===')
data.info()
print()
print('=== Estadísticas básicas ===')
data[FEATURES + [TARGET]].describe()